# Legumin hexamer: thermal and shear MD in GROMACS (Colab)

Complete workflow for the pea legumin hexamer (AlphaFold3 model, 6 chains of 496 residues): system setup, 1 ns production runs at 300 K, 373 K and 373 K with box shear (two replicates each), and the analysis of every run.

| Item | Setting |
|---|---|
| Software | GROMACS 2026.3 (conda-forge CUDA build), MDAnalysis, SciPy, pandas, Matplotlib |
| Hardware | Colab T4 GPU |
| Force field / water | AMBER99SB-ILDN / SPC/E |
| Replicates | r1 (seed 111) and r2 (seed 222) |

**How to use**

1. Run section 1 once. The runtime restarts after the `condacolab` cell; continue from the next cell.
2. In section 2, set `PROJECT` to a Drive folder of your own and put the AlphaFold3 model in `structures/legumin_hexamer_original.pdb` inside it.
3. Sections 3–4 build and minimise the system. Section 5 runs equilibration and production for one condition and replicate; change `T` and `R` at the top of the cell. Section 6 runs the shear production. Sections 7–8 analyse a run and compare all runs.

Settings for every stage are documented in `docs/methods.md`.

## 1. Install GROMACS 2026.3 (CUDA build)

In [ ]:
!nvidia-smi

The runtime restarts once after this cell. Continue with the next cell when it is back.

In [ ]:
%pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
%%bash
CONDA_OVERRIDE_CUDA=12.9 conda create -y \
    -n gmx_cuda \
    -c conda-forge \
    "gromacs=2026.3=nompi_cuda*"

## 2. Project setup
Mounts Drive, sets the project folder, and puts GROMACS on the path. Every later cell works inside `/content/pm`, a short alias for `PROJECT`.

In [ ]:
import os, subprocess
from google.colab import drive

drive.mount("/content/drive")

# Drive folder that holds all project files. Edit this path.
PROJECT = "/content/drive/MyDrive/legumin_md"
os.makedirs(f"{PROJECT}/structures", exist_ok=True)

# Short, space-free alias used by every later cell
os.system(f'ln -sfn "{PROJECT}" /content/pm')
os.chdir("/content/pm")

# GROMACS from the conda environment created in section 1
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
out = subprocess.run('gmx --version | grep -E "GROMACS version|GPU support|SIMD instructions"',
                     shell=True, capture_output=True, text=True)
print(out.stdout or out.stderr)

## 3. Starting structure
Checks the AlphaFold3 model: expected 6 chains (A–F) of 496 residues, no HETATM records.

In [ ]:
from collections import Counter, defaultdict

PDB_PATH = "/content/pm/structures/legumin_hexamer_original.pdb"

residues = defaultdict(dict)
record_counts = Counter()
atom_names = defaultdict(Counter)
other_residues = Counter()

with open(PDB_PATH) as f:
    for line in f:
        record = line[:6].strip()

        if record:
            record_counts[record] += 1

        if record == "ATOM":
            chain = line[21].strip() or "_"
            resid = (line[22:27].strip(), line[17:20].strip())
            atom_name = line[12:16].strip()

            residues[chain][resid] = True
            atom_names[chain][atom_name] += 1

        elif record == "HETATM":
            other_residues[line[17:20].strip()] += 1

print("PDB record types:", record_counts)
print("\nResidue numbering by chain:")

for chain in sorted(residues):
    numbers = [int(r[0].split()[0]) for r in residues[chain]]
    print(
        f"Chain {chain}: first={min(numbers)}, "
        f"last={max(numbers)}, unique residues={len(numbers)}"
    )

print("\nHETATM residue types:", other_residues or "None")
print("\nUnique atom names in chain A:")
print(sorted(atom_names["A"].keys()))

## 4. System build and energy minimisation
`pdb2gmx` → `editconf` → `solvate` → `genion` → energy minimisation. Each chain gets its own topology file (`-merge no`).

In [ ]:
%%bash
set -e
cd /content/pm
gmx pdb2gmx \
    -f structures/legumin_hexamer_original.pdb \
    -o structures/legumin_hexamer_processed.gro \
    -p topol.top \
    -i posre.itp \
    -ff amber99sb-ildn \
    -water spce \
    -chainsep id_or_ter \
    -merge no
echo; sed -n '/\[ molecules \]/,$p' topol.top

In [ ]:
%%bash
set -e
cd /content/pm
gmx editconf \
    -f structures/legumin_hexamer_processed.gro \
    -o structures/legumin_hexamer_box.gro \
    -c \
    -d 1.2 \
    -bt dodecahedron

In [ ]:
%%bash
set -e
cd /content/pm
gmx solvate \
    -cp structures/legumin_hexamer_box.gro \
    -cs spc216.gro \
    -o structures/legumin_hexamer_solv.gro \
    -p topol.top

Ions: neutralise the system and add NaCl to 0.15 M. Expected result: 231 Na⁺, 195 Cl⁻ and 56,682 waters (217,446 atoms in total). `grompp` needs `-maxwarn 1` because the solvated system is not yet neutral.

In [ ]:
%%bash
set -e
cd /content/pm
cat > ions.mdp << EOF
integrator = steep
nsteps     = 0
cutoff-scheme = Verlet
coulombtype = cutoff
rcoulomb   = 1.0
rvdw       = 1.0
pbc        = xyz
continuation = yes
EOF

gmx grompp -f ions.mdp -c structures/legumin_hexamer_solv.gro -p topol.top -o ions.tpr -maxwarn 1
echo SOL | gmx genion -s ions.tpr -o structures/legumin_hexamer_ions.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15
echo; sed -n '/\[ molecules \]/,$p' topol.top

Energy minimisation (steepest descent, `emtol` 1000 kJ mol⁻¹ nm⁻¹, up to 50,000 steps). The minimised structure `em.gro` is the starting point of every NVT run.

In [ ]:
from pathlib import Path

project = Path("/content/pm")
mdp = project / "em.mdp"

mdp.write_text("""; Energy minimization
integrator      = steep
emtol           = 1000.0
emstep          = 0.01
nsteps          = 50000

cutoff-scheme   = Verlet
nstlist         = 20
coulombtype     = PME
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz

constraints     = h-bonds
""")

print(f"Created: {mdp}")
print(mdp.read_text())

In [ ]:
%%bash
set -e
cd /content/pm
gmx grompp -f em.mdp -c structures/legumin_hexamer_ions.gro -p topol.top -o em.tpr
gmx mdrun -deffnm em -ntmpi 1 -ntomp 2 -pin off
echo; grep -E "converged to Fmax|Potential Energy|Maximum force" em.log
ls -lh em.gro em.tpr em.edr em.log

## 5. Equilibration and production (thermal runs)
One cell runs all three stages for a single condition and replicate. Set `T` (300 or 373) and `R` (1 or 2) at the top; the seed is `R × 111`.

| Stage | Length | Notes |
|---|---|---|
| NVT | 100 ps | Position restraints, velocities generated at `T`, V-rescale (Protein / Non-Protein) |
| NPT | 500 ps | Position restraints, C-rescale barostat (1 bar, tau-p 2.0 ps), `refcoord-scaling = com` |
| Production | 1 ns | No restraints, NVT, protein-only compressed trajectory every 1 ps |

The 373 K shear runs (section 6) start from the NPT end state of the 373 K run with the same replicate number.

In [ ]:
%%bash
set -e
cd /content/pm
T=300; R=1   # T: 300 or 373 | R: 1 or 2
SEED=$((R*111))
echo "T=$T rep=$R seed=$SEED"

COMMON="integrator = md
dt = 0.002
constraints = h-bonds
constraint-algorithm = lincs
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb = 1.0
rvdw = 1.0
DispCorr = EnerPres
tcoupl = V-rescale
tc-grps = Protein Non-Protein
tau-t = 0.1 0.1
ref-t = ${T} ${T}"

# NVT 100 ps
TAG=nvt_${T}_r${R}
cat > ${TAG}.mdp << EOF
define = -DPOSRES
${COMMON}
nsteps = 50000
continuation = no
pcoupl = no
gen-vel = yes
gen-temp = ${T}
gen-seed = ${SEED}
nstenergy = 500
nstlog = 500
nstxout-compressed = 5000
EOF
gmx grompp -f ${TAG}.mdp -c em.gro -r em.gro -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

# NPT 500 ps
PREV=${TAG}; TAG=npt_${T}_r${R}
cat > ${TAG}.mdp << EOF
define = -DPOSRES
${COMMON}
nsteps = 250000
continuation = yes
pcoupl = C-rescale
pcoupltype = isotropic
tau-p = 2.0
compressibility = 4.5e-5
ref-p = 1.0
refcoord-scaling = com
gen-vel = no
nstenergy = 500
nstlog = 500
nstxout-compressed = 5000
EOF
gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -r ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

# Production 1 ns (NVT, no restraints)
PREV=${TAG}; TAG=md_${T}_r${R}
cat > ${TAG}.mdp << EOF
${COMMON}
nsteps = 500000
continuation = yes
pcoupl = no
gen-vel = no
ld-seed = ${SEED}
nstenergy = 500
nstlog = 5000
nstxout-compressed = 500
compressed-x-grps = Protein
EOF
gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

## 6. Shear production (373 K)
Starts from the NPT end state of the 373 K thermal run of the same replicate (`npt_373_r<R>`). The `.mdp` is the thermal production file with one thermostat group (`tc-grps = System`) and a box deformation. The tilt rate is set from the box height so that the strain rate is 0.25 ns⁻¹ (2.5 × 10⁻⁴ ps⁻¹):

```
deform = 0 0 0 <2.5e-4 × box_y> 0 0     ; nm/ps
```

For the legumin box this gives 3.644 nm/ns.

In [ ]:
%%bash
set -e
cd /content/pm
T=373; R=1   # R: 1 or 2
SEED=$((R*111))
PREV=npt_${T}_r${R}; TAG=md_${T}_shear_r${R}
BY=$(tail -n 1 ${PREV}.gro | awk '{print $2}')
DEF=$(awk -v b=$BY 'BEGIN{printf "%.6f", 2.5e-4*b}')
echo "b_y = $BY nm  ->  deform b_x velocity = $DEF nm/ps"

cat > ${TAG}.mdp << EOF
integrator = md
dt = 0.002
nsteps = 500000
continuation = yes
constraints = h-bonds
constraint-algorithm = lincs
cutoff-scheme = Verlet
coulombtype = PME
rcoulomb = 1.0
rvdw = 1.0
DispCorr = EnerPres
tcoupl = V-rescale
tc-grps = System
tau-t = 0.1
ref-t = ${T}
pcoupl = no
deform = 0 0 0 ${DEF} 0 0
gen-vel = no
ld-seed = ${SEED}
nstenergy = 500
nstlog = 5000
nstxout-compressed = 500
compressed-x-grps = Protein
EOF

gmx grompp -f ${TAG}.mdp -c ${PREV}.gro -t ${PREV}.cpt -p topol.top -o ${TAG}.tpr
gmx mdrun -deffnm ${TAG} -cpt 5 -ntmpi 1 -ntomp 2 -pin off

If a Colab session ends before a stage finishes, restart that `mdrun` from its checkpoint (`-cpi <name>.cpt`). A restarted run is written as `.partNNNN` files; join them into one 0–1000 ps trajectory before analysis (see the restart notes in `docs/methods.md`).

## 7. Analysis of one run
Set the run at the top of the cell (`TEMP`, `RUN`, `TPR_NAME`, `TRAJ_NAME`, `DEFORM`) and run it once per run. It checks the trajectory (1001 frames, 0–1000 ps), prepares a protein-only whole-hexamer trajectory, and writes RMSD, Rg, SASA, H-bonds, RMSF and DSSP data, figures and CSV files to `analysis/legumin_<TEMP>_<RUN>/`.

Tilt rates: `DEFORM = 3.644` for the shear runs, `0` for thermal runs.

In [ ]:
import os, sys, json, itertools, subprocess
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import drive

# ---- choose the run to analyse ----
TEMP, RUN = "373", "r2"            # TEMP: "300" or "373" | RUN: "r1", "r2", "shear_r1", "shear_r2"
TPR_NAME  = "md_373_r2"            # run input / log name
TRAJ_NAME = "md_373_r2"            # protein-only trajectory (for a restarted run, the joined file)
NCH = 6; DEFORM = 0.0              # hexamer; DEFORM = tilt rate in nm/ns (3.644 for shear runs, 0 for thermal)
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
os.environ["PATH"] = "/usr/local/envs/gmx_cuda/bin:" + os.environ["PATH"]
if os.system("gmx --version > /dev/null 2>&1") != 0: raise SystemExit("GROMACS not found - run your GROMACS setup cell first, then run this cell again")
PM = "/content/pm"
if not os.path.isdir(PM): raise SystemExit("Run the project setup cell (section 2) first")
if TRAJ_NAME == TPR_NAME:          # single-part run: make sure it finished
    if os.path.exists(f"{PM}/{TRAJ_NAME}.part0002.xtc"): raise SystemExit(f"{TRAJ_NAME} was restarted (part0002 files exist): join the parts into one trajectory and set TRAJ_NAME to the joined file")
    if not os.path.exists(f"{PM}/{TPR_NAME}.log") or "Finished mdrun" not in open(f"{PM}/{TPR_NAME}.log").read(): raise SystemExit(f"{TPR_NAME}.log is missing or has no 'Finished mdrun' line: the run did not finish")
OUT = f"/content/pm/analysis/legumin_{TEMP}_{RUN}"; T = f"{TEMP} K {'shear' if DEFORM else 'thermal'} ({RUN.split('_')[-1]})"
os.environ.update(P="/content/pm", S=TPR_NAME, TRAJ=TRAJ_NAME, OUT=OUT)
for d in ["graphs","logs","frames","xvgs"]: os.makedirs(f"{OUT}/{d}", exist_ok=True)
subprocess.run([sys.executable,"-m","pip","install","-q","MDAnalysis","scipy"])
import MDAnalysis as mda
from MDAnalysis.coordinates.XTC import XTCReader
from MDAnalysis.lib.distances import minimize_vectors
from MDAnalysis.analysis.align import rotation_matrix
from scipy.spatial import cKDTree
from matplotlib.colors import ListedColormap
def sh(cmd, tail=0):
    r = subprocess.run(["bash","-c",cmd], capture_output=True, text=True)
    o = (r.stdout + r.stderr).strip().split("\n")
    print("\n".join(o[-tail:] if tail else o))

# ---- check files, mdp, and trajectory continuity
print("== files"); sh('ls -lh $P/$S.tpr $P/$TRAJ.xtc; grep -iE "ref[-_]t|deform" $P/$S.mdp')
rd = XTCReader(f"{OUT.rsplit('/analysis',1)[0]}/{TRAJ_NAME}.xtc")
times = np.array([ts.time for ts in rd]); dims = np.array([ts.dimensions.copy() for ts in rd])
print("frames:", len(times), "| time", times[0], "->", times[-1], "| natoms:", rd.n_atoms)
assert rd.n_atoms < 100000, "xtc is not protein-only (it contains the whole system)"
assert times[0] == 0 and abs(times[-1]-1000) < 1e-3 and len(times) == 1001 and np.all(np.diff(times) == 1), "trajectory not continuous 0-1000 ps"
print("box a,b,c start:", dims[0,:3].round(2), "end:", dims[-1,:3].round(2))
print("angles start:", dims[0,3:].round(2), "end:", dims[-1,3:].round(2))
if DEFORM:
    gd = DEFORM/(dims[0,1]/10); print(f"shear rate {gd:.4f} /ns -> expected rigid rotation {-0.5*gd*180/np.pi:.2f} deg/ns about z")
else: print("thermal run: box angles should stay 60/60/90")

# ---- protein tpr, whole, first frame
print("== whole")
sh(r'''cd "$OUT/frames"
printf "Protein\n" | gmx convert-tpr -s "$P/$S.tpr" -o protein.tpr 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f "$P/$TRAJ.xtc" -o whole2.xtc -pbc whole 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f whole2.xtc -o first_w.gro -dump 0 2>&1 | tail -1''')

# ---- chain detection from .gro
lines = open(f"{OUT}/frames/first_w.gro").read().split("\n")
n = int(lines[1]); body = lines[2:2+n]
resnum = np.array([int(l[0:5]) for l in body]); nm = [l[10:15].strip() for l in body]
starts = [0] + [i for i in range(1, n) if resnum[i] < resnum[i-1]]; ends = starts[1:] + [n]
nres = [len(set(resnum[s:e])) for s, e in zip(starts, ends)]
print("atoms:", n, "| chains:", len(starts), "| atoms/chain:", [e-s for s,e in zip(starts,ends)], "| res/chain:", nres)
assert len(starts) == NCH, f"expected {NCH} chains; check the chain-detection output above"
assert len(set(e-s for s,e in zip(starts,ends))) == 1, "chains differ in size; check the chain-detection output above"
NR = nres[0]; CH = ["ch"+"ABCDEF"[i] for i in range(NCH)]; G = {}; ranges = {}
for k, (s, e) in enumerate(zip(starts, ends)):
    r = range(s+1, e+1); c = CH[k]; ranges[c] = [s, e]
    G[c] = list(r); G[c+"_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G[c+"_CA"] = [i for i in r if nm[i-1]=="CA"]
r = range(1, n+1)
G["Protein"] = list(r); G["Protein_bb"] = [i for i in r if nm[i-1] in ("N","CA","C")]; G["Protein_CA"] = [i for i in r if nm[i-1]=="CA"]
with open(f"{OUT}/frames/chains.ndx","w") as f:
    for g, idx in G.items():
        f.write(f"[ {g} ]\n")
        for k in range(0, len(idx), 15): f.write(" ".join(f"{i:5d}" for i in idx[k:k+15]) + "\n")
json.dump(dict(ranges=ranges, nres=NR, natoms=n), open(f"{OUT}/frames/chains.json","w"))

# ---- keep chains together (triclinic-safe)
sl = [slice(*ranges[c]) for c in CH]
u = mda.Universe(f"{OUT}/frames/first_w.gro", f"{OUT}/frames/whole2.xtc")
with mda.Writer(f"{OUT}/frames/together.xtc", u.atoms.n_atoms) as W:
    for ts in u.trajectory[::5]:
        pos = u.atoms.positions.copy(); box = ts.dimensions.astype(np.float32); a0 = pos[sl[0]].mean(0)
        for s in sl[1:]:
            d = pos[s].mean(0) - a0
            dm = minimize_vectors(d[None,:].astype(np.float32), box)[0]; pos[s] -= (d - dm)
        u.atoms.positions = pos; W.write(u.atoms)

# ---- fit
print("== fit")
sh(r'''cd "$OUT/frames"
printf "Backbone\nProtein\n" | gmx trjconv -s protein.tpr -f together.xtc -o fit_cluster_skip5.xtc -fit rot+trans 2>&1 | tail -1
printf "Protein\n" | gmx trjconv -s protein.tpr -f fit_cluster_skip5.xtc -o first_cl.gro -dump 0 2>&1 | tail -1
rm -f whole2.xtc together.xtc
gmx check -f fit_cluster_skip5.xtc 2>&1 | grep -E "Last frame|^Step"''')

# ---- hexamer integrity check
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
pairs = list(itertools.combinations(range(NCH), 2)); worst = {p: 0.0 for p in pairs}; lonely = 0
for ts in u.trajectory:
    Pp = u.atoms.positions/10; ch = [Pp[s] for s in sl]
    m = {p: cKDTree(ch[p[0]]).query(ch[p[1]])[0].min() for p in pairs}
    for p in pairs: worst[p] = max(worst[p], m[p])
    for i in range(NCH):
        if min(m[p] for p in pairs if i in p) > 0.5: lonely += 1
print({"ABCDEF"[a]+"ABCDEF"[b]: round(float(v),2) for (a,b), v in worst.items()})
print("chains with no neighbour within 0.5 nm:", lonely)
assert lonely == 0, "a chain drifted away"

# ---- GROMACS analysis
print("== gmx analysis (slow)")
PAIRSTR = " ".join(f'"{a} {b}"' for a, b in itertools.combinations(CH, 2))
os.environ.update(CHS=" ".join(["Protein"]+CH), PAIRSTR=PAIRSTR)
sh(r'''cd "$OUT/frames"; X="$OUT/xvgs"; L="$OUT/logs"; T=fit_cluster_skip5.xtc
for c in $CHS; do
  printf "${c}_bb\n${c}_bb\n" | gmx rms  -s first_cl.gro -f $T -n chains.ndx -o $X/rmsd_$c.xvg -tu ns > $L/rmsd_$c.log 2>&1
  printf "${c}_CA\n${c}_CA\n" | gmx rmsf -s first_cl.gro -f $T -n chains.ndx -o $X/rmsf_$c.xvg -res   > $L/rmsf_$c.log 2>&1
  printf "$c\n"               | gmx gyrate -s protein.tpr -f $T -n chains.ndx -o $X/rg_$c.xvg          > $L/rg_$c.log 2>&1
  gmx sasa -s protein.tpr -f $T -n chains.ndx -surface "group Protein" -output "group $c" -o $X/sasa_$c.xvg -or $X/tmp_$c.xvg -tu ns > $L/sasa_$c.log 2>&1
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r "group $c" -t "group $c" -num $X/hbnum_$c.xvg -tu ns > $L/hbond_$c.log 2>&1
done
eval "for p in $PAIRSTR; do set -- \$p
  gmx hbond -s protein.tpr -f $T -n chains.ndx -r \"group \$1\" -t \"group \$2\" -num $X/hbinter_\$1_\$2.xvg -tu ns > $L/hbinter_\$1_\$2.log 2>&1
done"
mv $X/tmp_Protein.xvg $X/sasa_perres_Protein.xvg; rm -f $X/tmp_ch*.xvg
for f in rmsd_Protein rmsd_chA rmsf_chA rg_Protein sasa_chA hbnum_Protein hbinter_chA_chB; do echo $f $(tail -1 $X/$f.xvg); done''')

# ---- plots
X = f"{OUT}/xvgs"
cols = ["#d62728","#1f77b4","#2ca02c","#ff9896","#7fbfff","#98df8a"]
sty = {c: dict(color=cols[i], ls="-" if i%2==0 else ":", label="Chain "+c[-1]) for i, c in enumerate(CH)}
xvg = lambda f: np.loadtxt(f, comments=('#','@'))
def draw(out, title, xl, yl, items):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for x, y, s in items: ax.plot(x, y, lw=1.6, **s)
    ax.set(xlabel=xl, ylabel=yl, title=title); ax.legend(ncol=2); fig.tight_layout()
    fig.savefig(f"{OUT}/graphs/{out}.png"); plt.show()
chains = lambda pre, col=1, sc=1: [(xvg(f"{X}/{pre}_{c}.xvg")[:,0]/sc, xvg(f"{X}/{pre}_{c}.xvg")[:,col], s) for c, s in sty.items()]
def whole(pre, col=1, sc=1):
    a = xvg(f"{X}/{pre}_Protein.xvg"); return [(a[:,0]/sc, a[:,col], dict(color="grey", label="Whole hexamer"))]
for pre, yl, ttl, sc in (("rmsd","RMSD (nm)","Backbone RMSD",1), ("rg","Rg (nm)","Radius of gyration",1000),
                         ("sasa","SASA (nm²)","SASA",1), ("hbnum","H-bonds","Intra-chain H-bonds",1)):
    cc = 2 if pre == "sasa" else 1
    draw(f"{pre}_per_chain", f"{ttl} per chain, {T}", "Time (ns)", yl, chains(pre, cc, sc))
    draw(f"{pre}_protein", f"{ttl}, whole hexamer, {T}", "Time (ns)", yl, whole(pre, 1, sc))
draw("rmsf_per_chain", f"C-alpha RMSF per residue, {T}", "Residue", "RMSF (nm)", chains("rmsf"))
d = xvg(f"{X}/sasa_perres_Protein.xvg"); assert len(d) == NCH*NR, len(d)
draw("sasa_per_residue", f"SASA per residue, {T}", "Residue", "SASA (nm²)",
     [(np.arange(1,NR+1), d[i*NR:(i+1)*NR,1], s) for i, s in enumerate(sty.values())])
fig, ax = plt.subplots(figsize=(12,6), dpi=150)
for k, (a_, b_) in enumerate(itertools.combinations(CH, 2)):
    h = xvg(f"{X}/hbinter_{a_}_{b_}.xvg"); ax.plot(h[:,0], h[:,1], lw=1.2, color=plt.cm.tab10(k), label=f"{a_[-1]}–{b_[-1]}")
ax.set(xlabel="Time (ns)", ylabel="H-bonds", title=f"Inter-chain H-bonds, {T}"); ax.legend(ncol=3, fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/hbonds_inter_chain.png"); plt.show()

# ---- per-residue RMSD and H-bonds
u = mda.Universe(f"{OUT}/frames/first_cl.gro", f"{OUT}/frames/fit_cluster_skip5.xtc")
NA = u.atoms.n_atoms // NCH; cs = {c: slice(i*NA, (i+1)*NA) for i, c in enumerate(CH)}; resR = {}
for name, s in cs.items():
    chn = u.atoms[s]; bb = chn.select_atoms("name N CA C"); ca = chn.select_atoms("name CA")
    u.trajectory[0]; c0 = bb.positions.mean(0); bb0 = bb.positions - c0; ca0 = ca.positions - c0
    d2 = np.zeros(len(ca)); k = 0
    for ts in u.trajectory:
        c = bb.positions.mean(0); R, _ = rotation_matrix(bb.positions - c, bb0)
        d2 += (((ca.positions - c) @ R.T - ca0)**2).sum(1); k += 1
    resR[name] = np.sqrt(d2/k)/10
    np.savetxt(f"{X}/rmsd_perres_{name}.xvg", np.c_[np.arange(1,NR+1), resR[name]], fmt="%d %.6f")
draw("rmsd_per_residue", f"C-alpha RMSD per residue vs first frame, {T}", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), resR[c], s) for c, s in sty.items()])
draw("rmsd_per_residue_mean", f"Mean C-alpha RMSD per residue ({NCH} chains), {T}", "Residue", "RMSD (nm)",
     [(np.arange(1,NR+1), np.mean(list(resR.values()), axis=0), dict(color="grey", label="Mean of chains"))])

nmA = np.array([x.strip() for x in u.atoms.names]); rn = u.atoms.resnames
chain = np.arange(len(nmA)) // NA; ridx = u.atoms.resindices - chain*NR
assert ridx.min() == 0 and ridx.max() == NR-1
isH = np.array([x.startswith("H") for x in nmA])
P0 = u.atoms.positions/10; heavy = np.where(~isH)[0]; Hs = np.where(isH)[0]
don = heavy[cKDTree(P0[heavy]).query(P0[Hs])[1]]
kk = np.array([nmA[i][0] in "NO" for i in don]); Hs, don = Hs[kk], don[kk]
acc = np.array([i for i in range(len(nmA)) if nmA[i][0]=="O" or (nmA[i] in ("ND1","NE2") and rn[i].startswith("HI"))])
tot = np.zeros((NCH,NR)); inter = np.zeros((NCH,NR)); nb = []
for ts in u.trajectory:
    Pf = u.atoms.positions/10; k = 0
    for hi, lst in enumerate(cKDTree(Pf[Hs]).query_ball_tree(cKDTree(Pf[acc]), 0.25)):
        D_ = don[hi]
        for aj in lst:
            A = acc[aj]
            if A == D_ or (chain[A]==chain[D_] and ridx[A]==ridx[D_]): continue
            v1, v2 = Pf[Hs[hi]]-Pf[D_], Pf[A]-Pf[D_]; dist = np.linalg.norm(v2)
            if dist > 0.35 or v1@v2/(np.linalg.norm(v1)*dist) < np.cos(np.radians(30)): continue
            k += 1
            for Xa in (D_, A):
                tot[chain[Xa], ridx[Xa]] += 1
                if chain[D_] != chain[A]: inter[chain[Xa], ridx[Xa]] += 1
    nb.append(k)
nf = len(nb); print("mean H-bonds/frame:", round(float(np.mean(nb)),1))
for nmx, arr, ttl in (("hbperres",tot,"All H-bonds per residue"), ("hbperres_inter",inter,"Inter-chain H-bonds per residue")):
    for i, c in enumerate(CH): np.savetxt(f"{X}/{nmx}_{c}.xvg", np.c_[np.arange(1,NR+1), arr[i]/nf], fmt="%d %.5f")
    draw(nmx, f"{ttl}, {T}", "Residue", "H-bonds per frame", [(np.arange(1,NR+1), arr[i]/nf, s) for i, s in enumerate(sty.values())])

# ---- DSSP
print("== dssp")
sh(r'''cd "$OUT/frames"
printf "Protein\n" | gmx dssp -s protein.tpr -f fit_cluster_skip5.xtc -n chains.ndx -o "$OUT/xvgs/dssp_Protein.dat" > "$OUT/logs/dssp_Protein.log" 2>&1
cp "$OUT/xvgs/dssp_Protein.dat" "$OUT/xvgs/dssp_chA.dat"
head -c 80 "$OUT/xvgs/dssp_Protein.dat"; echo''')
a = np.array([list(l.rstrip("\n")) for l in open(f"{X}/dssp_Protein.dat") if l.strip()])
a = a[:, ~(a == "=").all(0)]; print("dssp shape:", a.shape); assert a.shape[1] == NCH*NR, a.shape
Dd = {c: a[:, i*NR:(i+1)*NR] for i, c in enumerate(CH)}; t = np.arange(len(a))*0.005
for name, ss, yl in (("helix","HGI","Helix residues (%)"), ("strand","EB","Strand residues (%)")):
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        ax.plot(t, 100*np.isin(Dd[c], list(ss)).mean(1), lw=1.6, **s)
        print(name, c, "mean %:", round(100*np.isin(Dd[c], list(ss)).mean(), 1))
    ax.set(xlabel="Time (ns)", ylabel=yl, title=f"{name.capitalize()} content vs time, {T}")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_time.png"); plt.show()
    fig, ax = plt.subplots(figsize=(12,5), dpi=150)
    for c, s in sty.items():
        f = np.isin(Dd[c], list(ss)).mean(0); ax.plot(np.arange(1,NR+1), f, lw=1.4, **s)
        np.savetxt(f"{X}/dssp_{name}_perres_{c}.xvg", np.c_[np.arange(1,NR+1), f], fmt="%d %.4f")
    ax.set(xlabel="Residue", ylabel=f"Fraction of frames in {name}", title=f"{name.capitalize()} per residue, {T}")
    ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_{name}_perres.png"); plt.show()
fig, ax = plt.subplots(figsize=(12,5), dpi=150)
for c, s in sty.items(): ax.plot(t, 100*(1-np.isin(Dd[c], list("HGIEB")).mean(1)), lw=1.6, **s)
ax.set(xlabel="Time (ns)", ylabel="Coil/turn residues (%)", title=f"Coil/turn content vs time, {T}")
ax.legend(ncol=2); fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_coil_time.png"); plt.show()
cmap = ListedColormap(["#eeeeee", "#d62728", "#1f77b4"])
code = lambda m: np.where(np.isin(m, list("HGI")), 1, np.where(np.isin(m, list("EB")), 2, 0))
fig, axs = plt.subplots(NCH, 1, figsize=(12,2.2*NCH+1), dpi=150, sharex=True)
for ax, (c, s) in zip(np.atleast_1d(axs), sty.items()):
    ax.imshow(code(Dd[c]), aspect="auto", cmap=cmap, vmin=0, vmax=2, origin="lower", extent=[1,NR,0,1])
    ax.set_ylabel("Time (ns)"); ax.set_title(s["label"], loc="left", fontsize=10)
np.atleast_1d(axs)[-1].set_xlabel("Residue"); fig.suptitle(f"DSSP map: red = helix, blue = strand, grey = coil/turn, {T}")
fig.tight_layout(); fig.savefig(f"{OUT}/graphs/dssp_map.png"); plt.show()

# ---- summary + all-states plots
def last(f, col=1): b = xvg(f); return b[len(b)//2:, col].mean()
h = len(t)//2; rows = []
for c in CH:
    rows.append(dict(chain=c, RMSD_nm=last(f"{X}/rmsd_{c}.xvg"), Rg_nm=last(f"{X}/rg_{c}.xvg"),
      SASA_nm2=last(f"{X}/sasa_{c}.xvg", 2), intra_Hbonds=last(f"{X}/hbnum_{c}.xvg"),
      helix_pct=100*np.isin(Dd[c][h:], list("HGI")).mean(), strand_pct=100*np.isin(Dd[c][h:], list("EB")).mean()))
rows.append(dict(chain="hexamer", RMSD_nm=last(f"{X}/rmsd_Protein.xvg"), Rg_nm=last(f"{X}/rg_Protein.xvg"),
                 SASA_nm2=last(f"{X}/sasa_Protein.xvg"), intra_Hbonds=last(f"{X}/hbnum_Protein.xvg")))
df = pd.DataFrame(rows).round(3); df.to_csv(f"{X}/summary_last0.5ns.csv", index=False); print(df)
STATES = [("Alpha helix","H"),("3-10 helix","G"),("Pi helix","I"),("Beta strand","E"),("Beta bridge","B"),
          ("Turn","T"),("Bend","S"),("Polyproline II helix","P")]
Pc = {k_: 100*(a == c_).mean(1) for k_, c_ in STATES}; Pc["Coil / Unstructured"] = 100 - sum(Pc.values())
tp = np.arange(len(a))*5.0
fig, ax = plt.subplots(figsize=(12,6.2), dpi=150)
for i, (k_, y) in enumerate(Pc.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.6, label=k_)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title=f"Legumin All Secondary Structure States — {T}")
ax.grid(alpha=.3); ax.legend(title="Secondary structure", ncol=2, loc="upper right"); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_all_states_{TEMP}K_{RUN}.png"); plt.show()
Gp = {"Alpha helix": Pc["Alpha helix"], "Beta structure": Pc["Beta strand"]+Pc["Beta bridge"], "Turns": Pc["Turn"]}
Gp["Loops / Other"] = 100 - sum(Gp.values())
fig, ax = plt.subplots(figsize=(10,6), dpi=150)
for i, (k_, y) in enumerate(Gp.items()): ax.plot(tp, y, color=plt.cm.tab10(i), lw=1.8, label=k_)
ax.set(xlabel="Time (ps)", ylabel="Percentage (%)", title=f"Legumin Secondary Structure — {T}")
ax.grid(alpha=.3); ax.legend(); fig.tight_layout()
fig.savefig(f"{OUT}/graphs/dssp_groups_{TEMP}K_{RUN}.png"); plt.show()
print("DONE ->", OUT)

## 8. Comparison of all runs
Loads the data of all six runs (300 K, 373 K and 373 K shear, r1 and r2) and writes the comparison tables (`csv/`) and figures (`graphs/`) to `analysis/comparison_legumin_with_r2/`.

In [ ]:
import os, itertools, numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount('/content/drive')
BASE = "/content/pm/analysis"
RUNS = {"300 K r1":"legumin_300_r1", "300 K r2":"legumin_300_r2",
        "373 K r1":"legumin_373_r1", "373 K r2":"legumin_373_r2",
        "373 K shear r1":"legumin_373_shear_r1", "373 K shear r2":"legumin_373_shear_r2"}
COL = {"300 K r1":"#1f77b4", "300 K r2":"#7fbfff", "373 K r1":"#d62728", "373 K r2":"#ff9896",
       "373 K shear r1":"#2ca02c", "373 K shear r2":"#98df8a"}
LS  = {r: ("-" if r.endswith("r1") else "--") for r in RUNS}
COND = {"300 K thermal":("300 K r1","300 K r2"), "373 K thermal":("373 K r1","373 K r2"),
        "373 K shear":("373 K shear r1","373 K shear r2")}
CMP = f"{BASE}/comparison_legumin_with_r2"
for d in ("graphs","csv"): os.makedirs(f"{CMP}/{d}", exist_ok=True)
NR = 496; NCH = 6; CH = ["chA","chB","chC","chD","chE","chF"]
PAIRS = list(itertools.combinations(CH, 2)); PN = [f"{a[-1]}-{b[-1]}" for a, b in PAIRS]
STAT = [("Alpha helix","H"),("3-10 helix","G"),("Pi helix","I"),("Beta strand","E"),
        ("Beta bridge","B"),("Turn","T"),("Bend","S"),("PPII helix","P")]
xvg = lambda f: np.loadtxt(f, comments=('#','@'))
ok = os.path.exists
def load_dssp(p):
    a = np.array([list(l.rstrip("\n")) for l in open(p) if l.strip()])
    return a[:, ~(a == "=").all(0)]
miss = [r for r, fd in RUNS.items() if not os.path.isdir(f"{BASE}/{fd}/xvgs")]
if miss: raise SystemExit(f"MISSING run folder(s): {miss} - analyse every run in section 7 first")

# ---------------- load everything ----------------
rows, resrows, comp = [], [], {}
def add(run, m, e, t, y): rows.extend((run, m, e, float(a), float(b)) for a, b in zip(t, y))
spec = {"RMSD (nm)":("rmsd",1,1.0), "Rg (nm)":("rg",1,1000.0),
        "SASA (nm2)":("sasa",2,1.0), "H-bonds (hexamer)":("hbnum",1,1.0)}
RESPAT = {"RMSF (nm)":"rmsf_{c}", "RMSD per residue (nm)":"rmsd_perres_{c}",
          "H-bonds per residue":"hbperres_{c}", "Inter-chain H-bonds per residue":"hbperres_inter_{c}",
          "Helix fraction":"dssp_helix_perres_{c}", "Strand fraction":"dssp_strand_perres_{c}"}
for run, fd in RUNS.items():
    X = f"{BASE}/{fd}/xvgs"
    for m, (pre, col, sc) in spec.items():
        for e in CH + ["Protein"]:
            f = f"{X}/{pre}_{e}.xvg"
            if ok(f): a = xvg(f); add(run, m, e, a[:,0]/sc, a[:,col])
            else: print("missing", run, f"{pre}_{e}.xvg")
    for (a_, b_), n in zip(PAIRS, PN):
        f = f"{X}/hbinter_{a_}_{b_}.xvg"
        if ok(f): a = xvg(f); add(run, "Inter H-bonds", n, a[:,0], a[:,1])
        else: print("missing", run, f"hbinter_{a_}_{b_}.xvg")
    f = next((p for p in (f"{X}/dssp_chA.dat", f"{X}/dssp_Protein.dat") if ok(p)), None)
    if f:
        a = load_dssp(f)
        if a.shape[1] != NCH*NR: print("DSSP shape mismatch, skipped:", run, a.shape)
        else:
            n = len(a); t = np.linspace(0, 1, n)
            ents = {"Protein": a, **{c: a[:, i*NR:(i+1)*NR] for i, c in enumerate(CH)}}
            for e, mm in ents.items():
                hel = 100*np.isin(mm, list("HGI")).mean(1); stn = 100*np.isin(mm, list("EB")).mean(1)
                add(run, "Helix (%)", e, t, hel); add(run, "Strand (%)", e, t, stn)
                add(run, "Coil/turn (%)", e, t, 100-hel-stn)
            comp[run] = {nm: 100*(a[n//2:] == ch).mean() for nm, ch in STAT}
            comp[run]["Coil/other"] = 100 - sum(comp[run].values())
    else: print("missing DSSP file", run)
    for c in CH:
        for m, pat in RESPAT.items():
            f = f"{X}/{pat.format(c=c)}.xvg"
            if ok(f):
                v = xvg(f)[:,1]
                if len(v) == NR: resrows.extend((run, m, c, i+1, float(x)) for i, x in enumerate(v))
    f = f"{X}/sasa_perres_Protein.xvg"
    if ok(f):
        d = xvg(f)
        if len(d) == NCH*NR:
            for i, c in enumerate(CH):
                resrows.extend((run, "SASA per residue (nm2)", c, j+1, float(x)) for j, x in enumerate(d[i*NR:(i+1)*NR, 1]))
ts  = pd.DataFrame(rows, columns=["run","metric","entity","time_ns","value"])
res = pd.DataFrame(resrows, columns=["run","metric","chain","residue","value"])
ts.to_csv(f"{CMP}/csv/timeseries_long.csv", index=False)
res.to_csv(f"{CMP}/csv/residue_metrics_long.csv", index=False)
pd.DataFrame(comp).T.round(2).to_csv(f"{CMP}/csv/dssp_composition_last_half.csv")

# ---------------- summaries ----------------
def stats(t, y):
    h = t >= t.max()/2
    return dict(mean_full=y.mean(), sd_full=y.std(ddof=1), mean_last_half=y[h].mean(), sd_last_half=y[h].std(ddof=1))
pc = [dict(run=r, metric=m, entity=e, **stats(d.sort_values("time_ns").time_ns.values, d.sort_values("time_ns").value.values))
      for (r, m, e), d in ts.groupby(["run","metric","entity"])]
pd.DataFrame(pc).round(4).to_csv(f"{CMP}/csv/summary_per_chain_and_interface.csv", index=False)
MET = [("RMSD (nm)",["Protein"],"mean","Backbone RMSD, whole hexamer (nm)"),
       ("Rg (nm)",["Protein"],"mean","Radius of gyration, whole hexamer (nm)"),
       ("SASA (nm2)",["Protein"],"mean","SASA, whole hexamer (nm²)"),
       ("H-bonds (hexamer)",["Protein"],"mean","H-bonds within the hexamer"),
       ("Inter H-bonds",PN,"sum","Inter-chain H-bonds (sum of 15 pairs)"),
       ("Helix (%)",["Protein"],"mean","Helix (%)"),
       ("Strand (%)",["Protein"],"mean","Strand (%)"),
       ("Coil/turn (%)",["Protein"],"mean","Coil/turn (%)")]
def curve(run, m, ents, agg):
    d = ts[(ts.run==run) & (ts.metric==m) & (ts.entity.isin(ents))]
    if d.empty: return None
    pr = [e for e in ents if (d.entity==e).any()]
    t0 = d[d.entity==pr[0]].sort_values("time_ns").time_ns.values
    Y = np.array([np.interp(t0, *[d[d.entity==e].sort_values("time_ns")[k].values for k in ("time_ns","value")]) for e in pr])
    return t0, (Y.sum(0) if agg=="sum" else Y.mean(0))
agg_rows = []
for run in RUNS:
    for m, ents, agg, yl in MET:
        c = curve(run, m, ents, agg)
        if c: agg_rows.append(dict(run=run, metric=m, **stats(*c)))
S = pd.DataFrame(agg_rows); S.round(4).to_csv(f"{CMP}/csv/summary_all_runs.csv", index=False)
table = S.pivot(index="run", columns="metric", values="mean_last_half").reindex(list(RUNS))
table = table[[m[0] for m in MET if m[0] in table.columns]]
table.round(4).to_csv(f"{CMP}/csv/last_half_means_by_run.csv")

# ---------------- conditions: replicate spread vs effect ----------------
crow = []
for m in table.columns:
    g = {c: (table.loc[a, m], table.loc[b, m]) for c, (a, b) in COND.items()}
    mean = {c: float(np.mean(v)) for c, v in g.items()}; gap = {c: abs(v[0]-v[1]) for c, v in g.items()}
    for name, hi, lo in (("thermal: 373 K - 300 K", "373 K thermal", "300 K thermal"),
                         ("shear: 373 K shear - 373 K thermal", "373 K shear", "373 K thermal")):
        dlt = mean[hi] - mean[lo]
        sep = bool(min(g[hi]) > max(g[lo]) or max(g[hi]) < min(g[lo]))
        crow.append(dict(metric=m, contrast=name, baseline=mean[lo], value=mean[hi], delta=dlt,
                         pct=100*dlt/mean[lo], gap_baseline=gap[lo], gap_value=gap[hi],
                         bigger_than_both_gaps=abs(dlt) > max(gap[lo], gap[hi]), replicates_separated=sep))
CT = pd.DataFrame(crow); CT.round(4).to_csv(f"{CMP}/csv/condition_contrasts.csv", index=False)

# ---------------- graphs: time series ----------------
def ts_ax(ax, m, ents, agg, yl):
    for run in RUNS:
        c = curve(run, m, ents, agg)
        if c: ax.plot(c[0], c[1], color=COL[run], ls=LS[run], lw=1.5, label=run)
    ax.set(xlabel="Time (ns)", ylabel=yl, title=yl); ax.grid(alpha=.3); ax.legend(fontsize=7)
fig, axs = plt.subplots(4, 2, figsize=(14, 17))
for ax, (m, ents, agg, yl) in zip(axs.ravel(), MET): ts_ax(ax, m, ents, agg, yl)
fig.suptitle("Legumin hexamer: all runs", y=1.0); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/00_overview_timeseries.png", dpi=150); plt.show()
for i, (m, ents, agg, yl) in enumerate(MET, 1):
    f2, a2 = plt.subplots(figsize=(10, 5), dpi=150); ts_ax(a2, m, ents, agg, yl); f2.tight_layout()
    f2.savefig(f"{CMP}/graphs/ts_{m.split(' (')[0].replace(' ','_').replace('/','_')}.png"); plt.close(f2)

# ---------------- graphs: bars ----------------
fig, axs = plt.subplots(2, 4, figsize=(18, 7))
for ax, (m, ents, agg, yl) in zip(axs.ravel(), MET):
    d = S[S.metric==m].set_index("run").reindex(list(RUNS)).dropna()
    ax.bar(d.index, d.mean_last_half, yerr=d.sd_last_half, color=[COL[r] for r in d.index], capsize=3)
    ax.set_title(m, fontsize=10); ax.tick_params(axis="x", rotation=40); ax.grid(axis="y", alpha=.3)
fig.suptitle("Last-half mean ± SD over time (whole hexamer)"); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/01_bar_summary.png", dpi=150); plt.show()
P = pd.DataFrame(pc)
fig, axs = plt.subplots(2, 3, figsize=(18, 8)); w = 0.13
for ax, m in zip(axs.ravel(), ["RMSD (nm)","Rg (nm)","SASA (nm2)","H-bonds (hexamer)","Helix (%)","Strand (%)"]):
    for k, run in enumerate(RUNS):
        d = P[(P.run==run)&(P.metric==m)&(P.entity.isin(CH))].set_index("entity").reindex(CH)
        ax.bar(np.arange(NCH)+k*w, d.mean_last_half, w, yerr=d.sd_last_half, color=COL[run], label=run, capsize=2)
    ax.set_xticks(np.arange(NCH)+2.5*w); ax.set_xticklabels([f"Chain {c[-1]}" for c in CH]); ax.set_title(m); ax.grid(axis="y", alpha=.3)
axs[0,0].legend(fontsize=7); fig.suptitle("Per-chain last-half mean ± SD"); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/02_per_chain_bars.png", dpi=150); plt.show()

# heatmap of % change vs 300 K baseline
base = table.loc[["300 K r1","300 K r2"]].mean(); pct = 100*(table - base)/base
fig, ax = plt.subplots(figsize=(12, 4.4)); v = np.nanpercentile(np.abs(pct.values), 95)
im = ax.imshow(pct.values, cmap="RdBu_r", vmin=-v, vmax=v, aspect="auto")
ax.set_xticks(range(pct.shape[1])); ax.set_xticklabels(pct.columns, rotation=30, ha="right")
ax.set_yticks(range(pct.shape[0])); ax.set_yticklabels(pct.index)
for i in range(pct.shape[0]):
    for j in range(pct.shape[1]): ax.text(j, i, f"{pct.values[i,j]:.1f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, label="% change vs mean of 300 K r1/r2"); ax.set_title("Change relative to the 300 K baseline (%)")
fig.tight_layout(); fig.savefig(f"{CMP}/graphs/03_pct_change_heatmap.png", dpi=150); plt.show()

# DSSP composition
if comp:
    C = pd.DataFrame(comp).T.reindex([r for r in RUNS if r in comp])
    ax = C.plot(kind="bar", stacked=True, figsize=(11, 5), colormap="tab20"); ax.set_ylabel("% of residues (last half)")
    ax.set_title("DSSP composition, whole hexamer"); ax.legend(ncol=2, fontsize=8, bbox_to_anchor=(1.01, 1))
    plt.xticks(rotation=25); plt.tight_layout(); plt.savefig(f"{CMP}/graphs/04_dssp_composition.png", dpi=150); plt.show()

# replicate dot plot: the two replicates of each condition side by side
fig, axs = plt.subplots(2, 4, figsize=(17, 8)); cn = list(COND)
for ax, m in zip(axs.ravel(), table.columns):
    for k, (c, (a_, b_)) in enumerate(COND.items()):
        ya, yb = table.loc[a_, m], table.loc[b_, m]
        ax.plot([k, k], [ya, yb], color="grey", lw=1)
        ax.scatter([k], [ya], color=COL[a_], s=60, zorder=3, label="r1" if k == 0 else None, edgecolor="k")
        ax.scatter([k], [yb], color=COL[b_], s=60, zorder=3, marker="s", label="r2" if k == 0 else None, edgecolor="k")
    ax.set_xticks(range(3)); ax.set_xticklabels([c.replace(" ", "\n", 1) for c in cn], fontsize=8)
    ax.set_xlim(-.5, 2.5); ax.set_title(m, fontsize=10); ax.grid(axis="y", alpha=.3)
axs[0,0].legend(fontsize=8)
fig.suptitle("Replicates per condition (last-half mean, whole hexamer): a real effect separates the pairs"); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/08_replicate_dotplot.png", dpi=150); plt.show()

# ---------------- graphs: per residue ----------------
rm = res.groupby(["run","metric","residue"]).value.mean()
def rcurve(run, m):
    try: s = rm.loc[(run, m)]; return s.index.values, s.values
    except KeyError: return None
def rmean(runs, m):
    cs = [rcurve(r, m) for r in runs]; cs = [c for c in cs if c]
    return (cs[0][0], np.mean([c[1] for c in cs], axis=0)) if cs else None
RM = list(RESPAT) + ["SASA per residue (nm2)"]
fig, axs = plt.subplots(4, 2, figsize=(15, 17))
for ax, m in zip(axs.ravel(), RM):
    for run in RUNS:
        c = rcurve(run, m)
        if c: ax.plot(*c, color=COL[run], ls=LS[run], lw=1.2, label=run)
    ax.set(xlabel="Residue", ylabel=m, title=m + " (mean of 6 chains)"); ax.grid(alpha=.3); ax.legend(fontsize=7)
axs.ravel()[-1].axis("off"); fig.tight_layout()
fig.savefig(f"{CMP}/graphs/05_residue_overlays.png", dpi=150); plt.show()
DM = ["RMSF (nm)","SASA per residue (nm2)","Helix fraction","Strand fraction","H-bonds per residue"]
fig, axs = plt.subplots(len(DM), 2, figsize=(15, 3.2*len(DM))); top = []
for i, m in enumerate(DM):
    a300, a373, ash = rmean(COND["300 K thermal"], m), rmean(COND["373 K thermal"], m), rmean(COND["373 K shear"], m)
    if not (a300 and a373 and ash): continue
    resid = a373[0]
    for j, (name, dl) in enumerate((("Thermal: 373 K mean - 300 K mean", a373[1]-a300[1]), ("Shear: 373 K shear mean - 373 K thermal mean", ash[1]-a373[1]))):
        ax = axs[i, j]; ax.fill_between(resid, dl, 0, where=dl>=0, color="#d62728", alpha=.6)
        ax.fill_between(resid, dl, 0, where=dl<0, color="#1f77b4", alpha=.6)
        ax.axhline(0, color="k", lw=.6); ax.set(xlabel="Residue", ylabel=f"Δ {m}", title=name); ax.grid(alpha=.3)
        for k in np.argsort(-np.abs(dl))[:10]: top.append(dict(contrast=name, metric=m, residue=int(resid[k]), delta=float(dl[k])))
fig.tight_layout(); fig.savefig(f"{CMP}/graphs/06_residue_deltas.png", dpi=150); plt.show()
pd.DataFrame(top).round(4).to_csv(f"{CMP}/csv/top10_residue_changes.csv", index=False)

# ---------------- report ----------------
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
print("\nLast-half means, whole hexamer:\n", table.round(3))
sh = CT[CT.contrast.str.startswith("shear")].set_index("metric")[["baseline","value","delta","pct","gap_baseline","gap_value","bigger_than_both_gaps","replicates_separated"]]
th = CT[CT.contrast.str.startswith("thermal")].set_index("metric")[["baseline","value","delta","pct","gap_baseline","gap_value","bigger_than_both_gaps","replicates_separated"]]
print("\nShear effect (373 K shear vs 373 K thermal), against replicate gaps:\n", sh.round(3))
print("\nThermal effect (373 K vs 300 K), against replicate gaps:\n", th.round(3))
print("\nSaved to:", CMP)